# FAPT-GNN Training on Google Colab

Train the Fragility-Aware Phase Transition GNN for financial crash prediction using Google Colab GPU.

**Expected training time**: ~5-10 minutes for 10 epochs with GPU

## Steps:
1. Install dependencies
2. Mount Google Drive
3. Copy project from Drive (or GitHub)
4. Run training
5. Download trained model

## Step 1: Check GPU and Install Dependencies

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

⚡ **OPTIMIZED INSTALLATION**: This notebook uses precompiled wheels instead of building from source. This saves ~5-10 minutes compared to standard PyTorch + PyG setup.

**Why**: 
- PyTorch CUDA binaries are pre-built (~2-3GB)
- PyG wheels from official server (no source compilation)
- Removed unused packages (torchvision, torchaudio, torch-scatter, torch-sparse)

**Expected time**: ~2-3 minutes instead of 10-15 minutes

In [ ]:
# Install required packages (OPTIMIZED for speed - no compilation needed)
print("Installing PyTorch with CUDA 11.8...")
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118 2>&1 | tail -1

print("Installing PyTorch Geometric with prebuilt wheels...")
!pip install -q torch_geometric -f https://data.pyg.org/whl/torch-2.4.0+cu118.html 2>&1 | tail -1

print("Installing other dependencies...")
!pip install -q pandas numpy scikit-learn pyyaml plotly yfinance networkx scipy 2>&1 | tail -1

print("[OK] All packages installed successfully!")
print("\nTime saved: ~5-10 minutes by using pre-compiled wheels instead of building from source")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")
print("\nContents of My Drive:")
os.system('ls -la /content/drive/MyDrive/ | head -20')

## Step 3: Clone/Copy Project from Drive

**Option A**: If you have frp1 folder in Google Drive

In [ ]:
# Option A: Copy from Google Drive (recommended)
!cp -r /content/drive/MyDrive/frp1 /content/frp1
os.chdir('/content/frp1')
print(f"Current directory: {os.getcwd()}")
print("\nProject structure:")
os.system('ls -la')

**Option B**: Clone from GitHub (if you have it there)

In [ ]:
# Option B: Clone from GitHub
# Uncomment if using GitHub instead of Drive
# !git clone https://github.com/yourusername/frp1.git /content/frp1
# os.chdir('/content/frp1')

## Step 4: Import Core Modules

Add project to path and import modules

In [ ]:
import sys
sys.path.insert(0, '/content/frp1')

import yaml
import time
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
from pathlib import Path

# Import project modules
from data.data_pipeline import load_all_data
from data.feature_engineering import build_all_features, build_node_feature_matrix, compute_returns
from data.crash_labeler import create_labels
from data.graph_builder import build_graph_sequence
from models.fapt_gnn import FAPT_GNN
from training.losses import FAPTGNNLoss
from training.trainer import build_sliding_window_dataset, walk_forward_split, train_epoch, eval_epoch, compute_pos_weight
from training.evaluate import Evaluator

print("[OK] All modules imported successfully!")

## Step 5: Load Configuration

Use fast config for quick training (~5-10 minutes per epoch on GPU)

In [ ]:
# Load FAST config for quick training (optimized for GPU)
CONFIG_PATH = "experiments/config_fast.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print(f"[OK] Loaded config from {CONFIG_PATH}")
print(f"\nConfig Summary:")
print(f"  Data period: {config['data']['start_date']} to 2026")
print(f"  Centrality window: {config['features']['centrality_window']} days")
print(f"  Model parameters: {config['model']['gnn_hidden_dim']} hidden dim")
print(f"  Training epochs: {config['training']['epochs']}")
print(f"  Stride: {config['training']['stride']} (sample every Nth window)")

## Step 6: Load Data

Download market data and prepare features

In [ ]:
print("[1] Loading market data (NIFTY 50, VIX, macro)...")
start_time = time.time()

data = load_all_data(start=config['data']['start_date'])

print(f"[OK] Data loaded in {time.time() - start_time:.1f}s")
print(f"  NIFTY 50 days: {len(data['nifty'])}")
print(f"  Number of stocks: {data['prices'].shape[1]}")
print(f"  Date range: {data['nifty'].index[0].date()} to {data['nifty'].index[-1].date()}")

## Step 7: Build Features

In [ ]:
print("[2] Building features...")
start_time = time.time()

# Create sentiment placeholder
sent = pd.DataFrame({
    'sentiment_diverge': np.zeros(len(data['vix'])),
    'vix_norm': (data['vix'].values - data['vix'].mean()) / (data['vix'].std() + 1e-8)
}, index=data['vix'].index)

# Build all features (this takes a few minutes - mainly centrality computation)
feats = build_all_features(
    data["prices"], data["vix"], data["macro"], sent, 
    **config['features']
)

print(f"[OK] Features built in {time.time() - start_time:.1f}s")

## Step 8: Create Labels

Generate crash labels for training

In [ ]:
print("[3] Creating crash labels...")
start_time = time.time()

returns = compute_returns(data["prices"])
labels = create_labels(
    data["nifty"], returns,
    percentile=config['labels']['percentile'],
    drawdown_threshold=config['labels']['drawdown_threshold'],
    forward_days=config['labels']['forward_days'],
    dd_window=config['labels']['dd_window']
)

print(f"[OK] Labels created in {time.time() - start_time:.1f}s")
crash_count = (labels['crash_label'] == 1).sum()
print(f"  Crash samples: {crash_count} / {len(labels)} ({100*crash_count/len(labels):.1f}%)")

## Step 9: Build Graph Sequence

In [ ]:
print("[4] Building graph sequence...")
start_time = time.time()

node_feats = build_node_feature_matrix(feats)
graph_sequence, _ = build_graph_sequence(
    node_feats, feats, sent, 
    window=config['graph']['graph_window']
)

print(f"[OK] Graph sequence built in {time.time() - start_time:.1f}s")
print(f"  Graph snapshots: {len(graph_sequence)}")
print(f"  Nodes per graph: {graph_sequence[0].num_nodes}")
print(f"  Edges in last graph: {graph_sequence[-1].edge_index.shape[1]}")

## Step 10: Prepare Dataset

In [ ]:
print("[5] Building training dataset...")
start_time = time.time()

dataset = build_sliding_window_dataset(
    graph_sequence, labels, data['vix'],
    seq_len=config['model']['seq_len'],
    stride=config['training']['stride']
)

train_ds, val_ds, test_ds = walk_forward_split(dataset)

print(f"[OK] Dataset prepared in {time.time() - start_time:.1f}s")

## Step 11: Initialize Model

In [ ]:
print("[6] Initializing model...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Device: {device}")

model = FAPT_GNN(**config['model'])
model = model.to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Parameters: {num_params:,}")
print(f"  Model device: {next(model.parameters()).device}")

# Loss and optimizer
pos_weight = compute_pos_weight(train_ds)
criterion = FAPTGNNLoss(pos_weight=pos_weight, **config['loss'])

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config['training']['lr'],
    weight_decay=config['training']['weight_decay']
)

scaler = torch.cuda.amp.GradScaler()

print("[OK] Model initialized")

## Step 12: TRAIN MODEL

This will take ~5-10 minutes for 10 epochs on GPU

In [ ]:
print("="*80)
print("STARTING TRAINING")
print("="*80)
print(f"Train samples: {len(train_ds)}")
print(f"Val samples: {len(val_ds)}")
print(f"Epochs: {config['training']['epochs']}")
print(f"Device: {device}")
print("="*80 + "\n")

total_start = time.time()
best_val_auc = 0.0

for epoch in range(1, config['training']['epochs'] + 1):
    epoch_start = time.time()
    
    # Training
    train_evaluator = Evaluator()
    train_losses = train_epoch(
        model, train_ds, criterion, optimizer, scaler, device, train_evaluator,
        max_grad_norm=config['training']['max_grad_norm']
    )
    
    # Validation
    val_evaluator = Evaluator()
    val_losses, val_metrics = eval_epoch(model, val_ds, criterion, device, val_evaluator)
    
    val_auc = val_metrics.get('auc', 0)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
    
    epoch_time = time.time() - epoch_start
    
    print(f"Epoch {epoch:2d} | Time: {epoch_time:6.1f}s | "
          f"Loss: {train_losses.get('total', 0):.4f} | "
          f"Val AUC: {val_auc:.4f}")

total_time = time.time() - total_start
print(f"\n{'='*80}")
print(f"Training complete!")
print(f"Total time: {total_time//60:.0f}m {total_time%60:.0f}s")
print(f"Best Val AUC: {best_val_auc:.4f}")
print(f"{'='*80}")

## Step 13: Save Trained Model

Save to Google Drive for download

In [ ]:
# Create checkpoint directory
os.makedirs("/content/frp1/experiments/checkpoints", exist_ok=True)

# Save checkpoint
checkpoint_path = "/content/frp1/experiments/checkpoints/best_model.pt"

torch.save({
    'epoch': config['training']['epochs'],
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'val_auc': best_val_auc,
    'config': config,
}, checkpoint_path)

print(f"[OK] Model saved to {checkpoint_path}")
print(f"File size: {os.path.getsize(checkpoint_path) / 1024**2:.2f} MB")

## Step 14: Copy to Google Drive for Download

In [ ]:
# Create output folder in Google Drive
drive_output = "/content/drive/MyDrive/frp1_trained_model"
os.makedirs(drive_output, exist_ok=True)

# Copy trained model to Drive
!cp /content/frp1/experiments/checkpoints/best_model.pt /content/drive/MyDrive/frp1_trained_model/

print(f"[OK] Model copied to Google Drive!")
print(f"\nLocation: {drive_output}")
print(f"\nTo download:")
print(f"  1. Right-click 'frp1_trained_model' folder in Google Drive")
print(f"  2. Select 'Download'")
print(f"  3. Extract and place 'best_model.pt' in your local project at:")
print(f"     c:\\Users\\HP\\OneDrive\\Desktop\\frp1\\experiments\\checkpoints\\best_model.pt")

## Step 15: Verify Model Works

Quick test to ensure model loads and predicts correctly

In [ ]:
print("Testing trained model...\n")

# Load model from checkpoint
loaded_model = FAPT_GNN(**config['model'])
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model = loaded_model.to(device)
loaded_model.eval()

print(f"[OK] Model loaded from checkpoint")

# Test on validation set
with torch.no_grad():
    sample = val_ds[0]
    graphs = [g.to(device) for g in sample['graphs']]
    
    crash_prob, tte_pred, instability, energy_seq, fragility_seq = loaded_model(graphs)
    
    print(f"\nTest Prediction:")
    print(f"  Crash Probability: {crash_prob.item():.4f}")
    print(f"  Time-to-Crash: {tte_pred.item():.2f} days")
    print(f"  Instability Index: {instability.item():.4f}")
    print(f"  System Energy: {energy_seq[-1].item():.6f}")
    print(f"\n[OK] Model works correctly!")

## FINAL INSTRUCTIONS: Using Trained Model Locally

### Where to Put the Model

After downloading `best_model.pt` from Google Drive:

1. **Create the directory** if it doesn't exist:
   ```
   c:\Users\HP\OneDrive\Desktop\frp1\experiments\checkpoints\
   ```

2. **Place the file**:
   ```
   c:\Users\HP\OneDrive\Desktop\frp1\experiments\checkpoints\best_model.pt
   ```

3. **Use in Dashboard** (automatically loads):
   ```bash
   cd c:\Users\HP\OneDrive\Desktop\frp1
   streamlit run dashboard.py
   ```
   The trained model will be automatically loaded!

### Expected Results with Trained Model

- **Tab 1 (Market Context)**: ~5% historical crash ratio
- **Tab 3 (Predictions)**: Real predictions (not 50%)
  - Low risk: 5-20%
  - Medium risk: 30-50%
  - High risk: 70-95%

### Testing the Model

Run diagnostic to verify everything works:
```bash
python diagnose_model.py
```

Should show:
- ✓ Checkpoint loaded
- ✓ Model initialized
- ✓ Predictions working
- ✓ Energy values non-zero